In [48]:
import pandas as pd

from pypdf import PdfReader
from pathlib import Path
import pdfplumber

In [49]:
data_folder = Path("../data/raw")

csv_files = list(data_folder.glob("*.csv"))
pdf_files = list(data_folder.glob("*.pdf"))

print("CSV files:")
for file in csv_files:
    print(file)

print("\nPDF files:")
for file in pdf_files:
    print(file)

CSV files:
..\data\raw\2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv
..\data\raw\2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv
..\data\raw\2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv

PDF files:
..\data\raw\11th-July-2025-Council-Minutes.pdf
..\data\raw\30th-April-2025- Council-Minutes.pdf
..\data\raw\Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
..\data\raw\Petauke-Town-Council-2026-Budget-2026.pdf
..\data\raw\Petauke-Town-Council-Stratplan_2019-23.pdf
..\data\raw\Petauke.Lusangazi-Joint-IDP-Final.pdf


In [17]:
csv_data = {}

for file in csv_files:
    df = pd.read_csv(file)
    csv_data[file.name] = df

print(csv_data.keys())

dict_keys(['2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv', '2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv', '2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv'])


In [20]:
pdf_data = {}

for file in pdf_files:
    reader = PdfReader(file)

    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    pdf_data[file.name] = text

print(pdf_data.keys())

dict_keys(['Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf', 'Petauke-Town-Council-2026-Budget-2026.pdf'])




INSPECTING "2026 BUDGET" AND "2025 OBB FINAL"

In [55]:
data_folder = Path("../data/raw") 
output_folder = Path("../data/processed") 
output_folder.mkdir(parents=True, exist_ok=True)

In [73]:
for pdf_file in pdf_files:

    if "2025-OBB" in pdf_file.name or "2026-Budget" in pdf_file.name:

        print(f"\nProcessing: {pdf_file.name}")

        extracted_tables = []

        with pdfplumber.open(pdf_file) as pdf:

            for page_num, page in enumerate(pdf.pages, start=1):

                tables = page.extract_tables()

                for table in tables:

                    if table:

                        df = pd.DataFrame(table)

                        # Remove empty rows and columns
                        df = df.fillna("")
                        df = df.map(
                            lambda cell: cell.strip()
                            if isinstance(cell, str)
                            else cell
                        )

                        df = df.loc[~(df == "").all(axis=1)]
                        df = df.loc[:, ~(df == "").all(axis=0)]

                        if not df.empty:
                            df["source_page"] = page_num
                            extracted_tables.append(df)

        # Save extracted tables
        if extracted_tables:

            final_df = pd.concat(
                extracted_tables,
                ignore_index=True
            )

            clean_name = (
                pdf_file.stem
                .lower()
                .replace("-", "_")
                .replace(".", "_")
            )

            output_file = (
                output_folder /
                f"db-unza26-csc4792-{clean_name}.csv"
            )

            final_df.to_csv(
                output_file,
                sep="|",
                index=False
            )

            print(f"Saved: {output_file}")
            print(f"Rows: {len(final_df)}")
            print(f"Columns: {len(final_df.columns)}")

        else:
            print("No tables found.")


Processing: Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv
Rows: 422
Columns: 9

Processing: Petauke-Town-Council-2026-Budget-2026.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv
Rows: 446
Columns: 9


In [ ]:
processed_files = list(output_folder.glob("*.csv"))

print("Processed CSV files:")

for file in processed_files:
    print(file)

In [ ]:
obb_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv",
    sep="|"
)

obb_dataset.head()

In [ ]:
budget_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv",
    sep="|"
)

budget_dataset


In [68]:
obb_dataset.columns = obb_dataset.iloc[0]

obb_dataset = obb_dataset.iloc[1:].reset_index(drop=True)

In [ ]:
obb_dataset

## 1. Inspecting Petauke Town Council Strategic Plan 2019–2023

This section inspects the Petauke Town Council Strategic Plan 2019–2023.

**Objectives of this inspection:**
- Load the PDF and extract its raw text.
- Identify the document's structure (sections, headings, tables).
- Examine content relevant to the dataset (projects, strategies, budgets).
- Note potential data quality issues (page headers/footers, formatting artifacts).

**Source file:** `data/raw/Petauke-Town-Council-Stratplan_2019-23.pdf`

In [19]:
from pathlib import Path
from pypdf import PdfReader

data_folder = Path("../data/raw")

stratplan_path = data_folder / "Petauke-Town-Council-Stratplan_2019-23.pdf"
stratplan_reader = PdfReader(str(stratplan_path))

stratplan_text = ""
for page in stratplan_reader.pages:
    stratplan_text += (page.extract_text() or "") + "\n"

print(f"File           : {stratplan_path.name}")
print(f"Pages          : {len(stratplan_reader.pages)}")
print(f"Total chars    : {len(stratplan_text):,}")

File           : Petauke-Town-Council-Stratplan_2019-23.pdf
Pages          : 72
Total chars    : 124,985


In [20]:
print(stratplan_text[:3000])

 STRATEGIC PLAN 2019-2023 
“A Clean, Healthy, Green Town that is connected and Inclusive of 
All in Local Economic Growth”
PETAUKE TOWN COUNCIL 

Table of Contents  
 
Table of Contents ...................................................................................................................................... ii 
Foreword .................................................................................................................................................. iii 
Preface .....................................................................................................................................................iv  
Acknowledgement ................................................................................................................................ .v 
Acronyms ................................................................................................................................................. vi 
List of Tables .............................

In [21]:
import re

lines = [l.strip() for l in stratplan_text.split("\n") if l.strip()]

# Numbered headings and ALL CAPS headings
headings = [
    l for l in lines
    if 3 < len(l) < 90 and (
        re.match(r"^\d+(\.\d+)*[\s\.]", l) or
        re.match(r"^[A-Z][A-Z\s&,/-]{6,}$", l) or
        re.match(r"^(Chapter|Section|Part)\s+\d+", l, re.IGNORECASE)
    )
]

print(f"Non-empty lines : {len(lines):,}")
print(f"Headings found  : {len(headings)}")
print("\nFirst 25 headings:")
for h in headings[:25]:
    print("   •", h)

Non-empty lines : 4,120
Headings found  : 695

First 25 headings:
   • PETAUKE TOWN COUNCIL
   • 3.3.5 Strategic Theme 5: Good Stewardship of the Natural and Built Environment  36
   • PETAUKE TOWN COUNCIL
   • FOREWORD
   • PETAUKE TOWN COUNCIL
   • PREFACE
   • PETAUKE TOWN COUNCIL
   • 5 day planning workshop. We salute your steadfastness and eagerness to implement
   • PETAUKE TOWN COUNCIL
   • PETAUKE TOWN COUNCIL
   • PETAUKE TOWN COUNCIL
   • 1.0 Introduction
   • 2013 and its effective devolution date 1st January 2015, The Constitution Amendment
   • PETAUKE TOWN COUNCIL
   • 1.1     Physiographic Characteristics
   • PETAUKE TOWN COUNCIL
   • 1.2     Socio-Economic Characteristics
   • 1.2.1 Population Characteristics
   • 49.3% men1. With a projected growth rate of 2.6% the current population for Petauke
   • 1 CSO: 2010-Census of popula on summary report
   • PETAUKE TOWN COUNCIL
   • 1 below showing urbanization of Eastern province which mirrors the urbanization of
   • 201

In [22]:
import pdfplumber

with pdfplumber.open(stratplan_path) as pdf:
    pages_with_tables = []
    total_tables = 0
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            total_tables += len(tables)
            pages_with_tables.append((i + 1, len(tables)))

print(f"Total pages         : {len(pdf.pages)}")
print(f"Total tables found  : {total_tables}")
print(f"Pages with tables   : {pages_with_tables[:15]}")

Total pages         : 72
Total tables found  : 179
Pages with tables   : [(1, 1), (2, 2), (3, 2), (4, 2), (5, 2), (6, 2), (7, 2), (8, 2), (9, 2), (10, 2), (11, 3), (12, 3), (13, 3), (14, 2), (15, 5)]


## 2. Inspecting Petauke–Lusangazi Joint Integrated Development Plan

This section inspects the Petauke–Lusangazi Joint Integrated Development Plan (IDP).

**Objectives of this inspection:**
- Load the IDP PDF and extract its raw text.
- Identify sections and tables that describe development projects and wards.
- Examine content relevant to the dataset (community projects, budget allocations).
- Note potential data quality issues.

**Source file:** `data/raw/Petauke.Lusangazi-Joint-IDP-Final.pdf`

In [23]:
idp_path = data_folder / "Petauke.Lusangazi-Joint-IDP-Final.pdf"
idp_reader = PdfReader(str(idp_path))

idp_text = ""
for page in idp_reader.pages:
    idp_text += (page.extract_text() or "") + "\n"

print(f"File           : {idp_path.name}")
print(f"Pages          : {len(idp_reader.pages)}")
print(f"Total chars    : {len(idp_text):,}")

File           : Petauke.Lusangazi-Joint-IDP-Final.pdf
Pages          : 181
Total chars    : 274,372


In [24]:
print(idp_text[:3000])

1 
 
 
REPUBLIC OF ZAMBIA 
MINSITRY OF LOCAL GOVERNMENT 
 
JOINT INTEGRATED DEVELOPMENT PLAN  
 
FOR 
 
PETAUKE/LUSANGAZI DISTRICTS 
 
“Improved Social and Economic Welfare Through Well -Coordinated 
Climate Smart Investments and Sustainable Development By 2030.” 
 
PLANNING SURVEY AND KEY ISSUES REPORT, 
DEVELOPMENT FRAMEWORK AND IMPLEMENTATION 
PLAN 
 
Prepared By: IDP Technical Team  
 PETAUKE / LUSANGAZI  
TOWN COUNCILS 
 
 
2 
 
FOREWORD 
The development of the Joint Integrated 
Development Plan (IDP) for both Petauke and 
Lusangazi Districts gives the critical opportunities 
for the people of Petauke and Lusangazi to define 
their own destinies in terms of development.  
The participartory approach provided the platform for the people of the two districts to come 
up with pertinent issue s, develop projects and programmes with the Implementation plan 
which will make the two districts develop in pratical terms and at an accelerated rate. 
We congratulate the IDP technical team, t

In [25]:
lines = [l.strip() for l in idp_text.split("\n") if l.strip()]

headings = [
    l for l in lines
    if 3 < len(l) < 90 and (
        re.match(r"^\d+(\.\d+)*[\s\.]", l) or
        re.match(r"^[A-Z][A-Z\s&,/-]{6,}$", l) or
        re.match(r"^(Chapter|Section|Part)\s+\d+", l, re.IGNORECASE)
    )
]

print(f"Non-empty lines : {len(lines):,}")
print(f"Headings found  : {len(headings)}")
print("\nFirst 25 headings:")
for h in headings[:25]:
    print("   •", h)

Non-empty lines : 10,035
Headings found  : 998

First 25 headings:
   • REPUBLIC OF ZAMBIA
   • MINSITRY OF LOCAL GOVERNMENT
   • JOINT INTEGRATED DEVELOPMENT PLAN
   • PETAUKE/LUSANGAZI DISTRICTS
   • PLANNING SURVEY AND KEY ISSUES REPORT,
   • DEVELOPMENT FRAMEWORK AND IMPLEMENTATION
   • PETAUKE / LUSANGAZI
   • TOWN COUNCILS
   • FOREWORD
   • ACKNOWLEDGEMENT
   • 1. The District Commissioners for Petauke and Lusangazi
   • 2. The Petauke Town Council
   • 3. The Lusangazi Town Council
   • 4. The IDP Technical Team
   • 5. The combined DDCC members for both Petauke and Lusangazi
   • 6. Their Royal Highnesses  which include Senior Chief Kalindawalo, Chief Mumbi,
   • 7. The communities of both Petauke and Lusangazi
   • 8. Petauke District Land Alliance
   • PETAUKE        LUSANGAZI
   • APPROVAL BY THE MINISTER OF LOCAL GOVERNMENT
   • 2.4 The Impact of the Continuation of Existing Population Trends on Land Use and Spatial
   • 3.2.5 The Impact of the Continuation of Existing Tre

In [26]:
with pdfplumber.open(idp_path) as pdf:
    pages_with_tables = []
    total_tables = 0
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            total_tables += len(tables)
            pages_with_tables.append((i + 1, len(tables)))

print(f"Total pages         : {len(pdf.pages)}")
print(f"Total tables found  : {total_tables}")
print(f"Pages with tables   : {pages_with_tables[:15]}")

Total pages         : 181
Total tables found  : 287
Pages with tables   : [(15, 3), (16, 2), (17, 16), (19, 1), (22, 3), (23, 3), (27, 1), (33, 1), (42, 1), (43, 1), (56, 2), (65, 2), (66, 3), (67, 1), (71, 3)]
